# EDA

In [6]:
import pandas as pd

df = pd.read_csv("../data/raw/sleep_health_and_lifestyle_dataset.csv")
df.shape

(374, 13)

In [7]:
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


In [8]:
df.dtypes

Person ID                    int64
Gender                         str
Age                          int64
Occupation                     str
Sleep Duration             float64
Quality of Sleep             int64
Physical Activity Level      int64
Stress Level                 int64
BMI Category                   str
Blood Pressure                 str
Heart Rate                   int64
Daily Steps                  int64
Sleep Disorder                 str
dtype: object

## Missing values

In [9]:
df.isna().sum()

Person ID                    0
Gender                       0
Age                          0
Occupation                   0
Sleep Duration               0
Quality of Sleep             0
Physical Activity Level      0
Stress Level                 0
BMI Category                 0
Blood Pressure               0
Heart Rate                   0
Daily Steps                  0
Sleep Disorder             219
dtype: int64

`Sleep Disorder` is empty for ~58% of rows. It's empty
for people who don't have a diagnosed disorder, so `prepare.py`
imputes it as `"Healthy"` instead of dropping those rows.

## Duplicate rows

In [10]:
df.duplicated().sum()

np.int64(0)

## Target class balance

In [11]:
df["Sleep Disorder"].value_counts(dropna=False)

Sleep Disorder
NaN            219
Sleep Apnea     78
Insomnia        77
Name: count, dtype: int64

## Inconsistent category labels

In [12]:
df["BMI Category"].value_counts()

BMI Category
Normal           195
Overweight       148
Normal Weight     21
Obese             10
Name: count, dtype: int64

`"Normal"` and `"Normal Weight"` are the same category written two
different ways — `prepare.py` merges them into `"Normal"`.

## Numeric columns summary

In [13]:
df[["Age", "Sleep Duration", "Heart Rate", "Daily Steps"]].describe()

,Age,Sleep Duration,Heart Rate,Daily Steps
count,374.000000,374.000000,374.000000,374.000000
mean,42.184492,7.132086,70.165775,6816.844920
std,8.673133,0.795657,4.135676,1617.915679
min,27.000000,5.800000,65.000000,3000.000000
25%,35.250000,6.400000,68.000000,5600.000000
50%,43.000000,7.200000,70.000000,7000.000000
75%,50.000000,7.800000,72.000000,8000.000000
max,59.000000,8.500000,86.000000,10000.000000


## Outliers in Heart Rate (IQR rule)

In [14]:
q1, q3 = df["Heart Rate"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["Heart Rate"] < low) | (df["Heart Rate"] > high)]
print(f"bounds: [{low}, {high}]")
print(f"{len(outliers)} rows flagged as outliers")
outliers[["Heart Rate"]].describe()

bounds: [62.0, 78.0]
15 rows flagged as outliers


,Heart Rate
count,15.000000
mean,83.000000
std,2.203893
min,80.000000
25%,81.000000
50%,83.000000
75%,85.000000
max,86.000000


## Summary

- 374 rows, 13 columns, no duplicate rows.
- `Sleep Disorder`: 219 missing → imputed as `"Healthy"`.
- `BMI Category`: `"Normal"` / `"Normal Weight"` merged into one label.
- `Heart Rate`: 15 rows outside the IQR range → dropped as outliers.

These are exactly the steps implemented in `code/datasets/prepare.py`.